# ERA5-Land in the Magdalena River Basin (daily): dynamical system properties

Time delay, embedding dimension (false nearest neighbours), Lyapunov exponents and Hurst exponent for the local gauge stations and for every ERA5-Land pixel in the basin. The calculations live in `snlda.dynamics`.

## 0. Setup

In [ ]:
from pathlib import Path

DATA = Path("Data")
RESULTS = Path("Results")

STATIONS_CSV = DATA / "Local CSV" / "Precipitation 1980-2020 data filled 1.csv"
COORDS_CSV = DATA / "Local CSV" / "Precipitation 1980-2020 coord filled.csv"
IDW_NC = DATA / "Local IDW" / "IDW_daily_010.nc"
ERA_NC = DATA / "ERALand NC" / "precipitation_eraland_d2.nc"
BASIN_NC = DATA / "Magdalena" / "GIS" / "Basin" / "Basin_ext_010.nc"

import numpy as np

from snlda import dynamics, grid, io

## 1. Load data

In [ ]:
stations, coords = io.read_station_data(STATIONS_CSV, COORDS_CSV)
era = io.read_netcdf_grid(ERA_NC)
era.data[era.data < 0] = np.nan        # no-data values: precipitation cannot be negative
basin = io.read_basin_mask(BASIN_NC)

## 2. Local gauge stations

In [ ]:
def show(k, total, name):
    print(f"{k + 1} of {total}: {name}")

# dimension_offset=0 reproduces the original notebooks (see the README); use 1 for the real dimension
table = dynamics.dynamics_table(stations[coords.index], progress=show, dimension_offset=0)
station_results = coords.drop(columns="Length").join(table)

out_dir = RESULTS / "Dynamical system properties"
out_dir.mkdir(parents=True, exist_ok=True)
station_results.to_csv(out_dir / "local_stations_daily.csv")

## 3. ERA5-Land pixels

In [ ]:
era_basin = grid.select_on_grid(era.data, era.lat, era.lon, basin.lat, basin.lon)

pixel_results = dynamics.dynamics_on_grid(era_basin, basin.mask, progress=show, dimension_offset=0)

In [ ]:
io.write_netcdf_grid(
    out_dir / "Eraland_daily.nc",
    pixel_results,
    lat=basin.lat,
    lon=basin.lon,
    title="eraland",
)